# Common Libraries

In [48]:
import os, sys, optax
import numpy as np
import equinox as eqx
from tqdm import tqdm, trange
import matplotlib.pylab as plt
from typing import Tuple, Dict

"""
Setting for GPU acceleration
"""
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false" # limit jax and TF from consuming all GPU memory

# Jax
import jax
jax.config.update("jax_enable_x64", True)
from jax import numpy as jnp
from jaxtyping import Float, Array

# Mujoco
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Tools/biomechanics/MonocularBiomechanics")
from monocular_demos.biomechanics_mjx.forward_kinematics import ForwardKinetics
from monocular_demos.biomechanics_mjx.monocular_trajectory import KineticsWrapper

In [53]:
def loss(
    model: KineticsWrapper,
    x: Float[Array, "times"],
    y: Float[Array, "times keypoints 3"],
    site_offset_regularization = 1e-1
) -> Tuple[Float, Dict]:

    timestamps = x
    keypoints3d = y
    metrics = {}

    # NOTE: steps is an make sure this retraces for different dimensions
    (state, constraints, next_states), (ang, vel, action), _ = model(
        timestamps,
        skip_vel=True,
        skip_action=True,
    )

    pred_kp3d = state.site_xpos

    l = jnp.mean((pred_kp3d - keypoints3d) ** 2) * 100 # so in cm
    metrics["kp_err"] = l

    # regularize marker offset
    l_site_offset = jnp.sum(jnp.square(model.site_offsets))
    l += l_site_offset * site_offset_regularization

    # make loss the first key in the dictionary by popping and building a new dictionary with the rest
    metrics = {"loss": l, **metrics}

    return l, metrics

@eqx.filter_jit
def step(model, opt_state, data, loss_grad, optimizer, **kwargs):
    x, targets = data

    (val, metrics), grads = loss_grad(model, x=x, y=targets, **kwargs)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates) # Update model with preserving immutable property
    
    return val, model, opt_state, metrics

# Params

In [3]:
model_path = "/home/seojin/Tools/biomechanics/MonocularBiomechanics/monocular_demos/biomechanics_mjx/data/humanoid/humanoid_torque.xml"
pose3d_path = "/mnt/ext1/seojin/temp/scale/pose3d.npy"

# Make model

In [4]:
fk = ForwardKinetics(xml_path = model_path)
actuator_names = [ mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(fk.model.nu) ]
body_names = [mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(fk.model.nbody)]
site_names = [mujoco.mj_id2name(fk.model, mujoco.mjtObj.mjOBJ_SITE, i) for i in range(fk.model.nsite)]

mappings = {
    "pelvis": ["pelvis"],
    "left_thigh": ["femur_l"],
    "left_leg": ["tibia_l", "talus_l", "calcn_l", "toes_l"],
    "right_thigh": ["femur_r"],
    "right_leg": ["tibia_r", "talus_r", "calcn_r", "toes_r"],
    "left_arm": ["humerus_l", "radius_l", "ulna_l", "hand_l"],
    "right_arm": ["humerus_r", "radius_r", "ulna_r", "hand_r"],
    "overall": body_names,
}

# Scale mix
scale_mixer = jnp.array([
    [1.0 if body in child_segments else 0.0 
     for segment, child_segments in mappings.items()]
    for body in body_names
])

"""
KineticsWrapper consists of a neural network and a forward kinematics module, 
    where the scaling factor for each body segment is computed from body_scale and scale_mixer

Property
    - NonLearnable
        - scale_mixer

    - Learnable
        - body_scale
        - site_offsets: offset of each site
"""
model = KineticsWrapper(fk, scale_mixer = scale_mixer)

/tmp/ipykernel_4045390/4056126199.py:25: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  model = KineticsWrapper(fk, scale_mixer = scale_mixer)


# Load data & Preprocessing

In [5]:
pose3d = np.load(pose3d_path)

# convert pose to m
pose = pose3d
pose = pose[:, :, [0, 2, 1]]
pose[:, :, 2] *= -1
pose /= 1000.0

pose = pose - np.min(pose, axis=1, keepdims=True)

timestamps = jnp.arange(len(pose)) / 30.0

dataset = (timestamps, pose)

# Calculate loss

In [18]:
results = model(timestamps, skip_vel = True, skip_action = True)

kinetics_output = results[0]
trajectory_output = results[1]

state = kinetics_output[0]
constraints = kinetics_output[1]
next_states = kinetics_output[2]

joint_angle = trajectory_output[0]
joint_velocity = trajectory_output[1]
action = trajectory_output[2]

In [19]:
pred_kp3d = state.site_xpos
correct_kp3d = pose

In [20]:
site_offset_regularization = 1e-1

In [21]:
l = jnp.mean((pred_kp3d - correct_kp3d) ** 2) * 100 # so in cm

metrics = {}
metrics["kp_err"] = l

# regularize marker offset
l_site_offset = jnp.sum(jnp.square(model.site_offsets))
l += l_site_offset * site_offset_regularization

In [22]:
# make loss the first key in the dictionary by popping and building a new dictionary with the rest
metrics = {"loss": l, **metrics}

# Fit model

In [36]:
lr_end_value = 1e-8
lr_init_value = 1e-4
max_iters: int = 5000
clip_by_global_norm = 0.1

In [34]:
# work out the transition steps to make the desired schedule
transition_steps = 10
lr_decay_rate = (lr_end_value / lr_init_value) ** (1.0 / (max_iters // transition_steps))
learning_rate = optax.warmup_exponential_decay_schedule(
    init_value=0,
    warmup_steps=0,
    peak_value=lr_init_value,
    end_value=lr_end_value,
    decay_rate=lr_decay_rate,
    transition_steps=transition_steps,
)

In [37]:
optimizer = optax.chain(
    optax.adamw(learning_rate=learning_rate, b1=0.8, weight_decay=1e-5),
    optax.zero_nans(),
    optax.clip_by_global_norm(clip_by_global_norm)
)

In [40]:
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

In [50]:
loss_grad = eqx.filter_value_and_grad(loss, has_aux=True)

In [62]:
counter = trange(max_iters)
for i in counter:
    # val, model, opt_state, metrics = step(model, opt_state, dataset, loss_grad, optimizer)
    
    x, targets = dataset
    (loss, metrics), gradients = loss_grad(model, x=x, y=targets) # metrics: auxiliary data
    updates, opt_state = optimizer.update(gradients, opt_state, eqx.filter(model, eqx.is_inexact_array))
    model = eqx.apply_updates(model, updates)

    # Print optimization result
    if i > 0 and i % int(max_iters // 10) == 0:
        print(f"iter: {i} loss: {loss}.")  # metrics: {metrics}")

    if i % 50 == 0:
        metrics = {k: v.item() for k,v in metrics.items()}
        print(loss, metrics)
    break

  0%|          | 0/5000 [00:00<?, ?it/s]2026-08-18 20:34:58.547005: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-08-18 20:34:59.363771: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-08-18 20:35:00.090470: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-08-18 20:35:00.6

5.379027286143912 {'kp_err': 5.342070887771297, 'loss': 5.379027286143912}
